# Fine-tune Vietnamese-biencoder with train & validation set (9:1 in full train dataset).

In [1]:
!pip install py_vncorenlp sentence-transformers

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 38.7 MB/s eta 0:00:0000:01
  Created wheel for py_vncorenlp: filename=py_vncorenlp-0.1.4-py3-none-any.whl size=4304 sha256=c3c7530bb1204abc2a3e2ba43194708b8a8a2d3290732cea023e9c6e766f3f2d
  Stored in directory: /root/.cache/pip/wheels/db/e5/ff/f4a1b4ece36e8582db1ca71150a34e987e65df50c35974e9bb
Successfully built py_vncorenlp


In [2]:
import json
import numpy as np
import pandas as pd
import py_vncorenlp

In [3]:
py_vncorenlp.download_model(save_dir='/kaggle/working')

--2026-03-09 06:24:54--  https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27412703 (26M) [application/octet-stream]
Saving to: ‘VnCoreNLP-1.2.jar’

     0K .......... .......... .......... .......... ..........  0% 11.0M 2s
    50K .......... .......... .......... .......... ..........  0% 27.0M 2s
   100K .......... .......... .......... .......... ..........  0% 16.4M 2s
   150K .......... .......... .......... .......... ..........  0% 45.9M 1s
   200K .......... .......... .......... .......... ..........  0%  119M 1s
   250K .......... .......... .......... .......... ..........  1% 18.0M 1s
   300K .......... .......... .......... .......... ..........  1%  121M 1s
   350K ..

In [4]:
# Load the word and sentence segmentation component
rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir='/kaggle/working')

text = "Ông Nguyễn Khắc Chúc  đang làm việc tại Đại học Quốc gia Hà Nội. Bà Lan, vợ ông Chúc, cũng làm việc tại đây."

output = rdrsegmenter.word_segment(text)

print(output)
# ['Ông Nguyễn_Khắc_Chúc đang làm_việc tại Đại_học Quốc_gia Hà_Nội .', 'Bà Lan , vợ ông Chúc , cũng làm_việc tại đây .']

2026-03-09 06:25:01 INFO  WordSegmenter:24 - Loading Word Segmentation model
['Ông Nguyễn_Khắc_Chúc đang làm_việc tại Đại_học Quốc_gia Hà_Nội .', 'Bà Lan , vợ ông Chúc , cũng làm_việc tại đây .']


In [5]:
# segment laws in corpus
corpus_Id2Text = {}
corpus_Text2Id = {}


with open("/kaggle/input/datasets/duongquanganh/chunked-corpus/chunked_corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

for raw in corpus:
    corpus_Id2Text[raw['chunk_id']] = raw['content_Article']
    corpus_Text2Id[raw['content_Article']] = raw['chunk_id']

In [6]:
with open("/kaggle/input/traindata-retrieval/train.json", "r", encoding = "utf-8") as f:
    train_data = json.load(f)

# segment question in train data
for q in train_data:
    output = rdrsegmenter.word_segment(q['question'])
    q['question'] = " ".join(output)

In [7]:
train = {'anchor': [], 'positive': []}
for q in train_data:
    for rel_law in q['relevant_laws']:
        pattern_rel_law = str(rel_law) + '_'
        i = 0
        while True:
            chunked_rel_law = pattern_rel_law + str(i)
            #print(chunked_rel_law)
            if chunked_rel_law not in corpus_Id2Text:
                break
            train['anchor'].append(q['question'])
            train['positive'].append(corpus_Id2Text[chunked_rel_law])
            i+=1

In [8]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers
from transformers import EarlyStoppingCallback
from datasets import Dataset

train_dataset = Dataset.from_dict(train)
split = train_dataset.train_test_split(test_size = 0.1)

train_dataset = split['train']
val_dataset = split['test']

2026-03-09 06:25:25.150116: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773037525.492983      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773037525.611445      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773037526.443244      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773037526.443284      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773037526.443287      55 computation_placer.cc:177] computation placer alr

In [9]:
model = SentenceTransformer('bkai-foundation-models/vietnamese-bi-encoder',device='cuda')
loss = MultipleNegativesRankingLoss(model)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

In [10]:
import torch
torch.cuda.empty_cache()

In [12]:
args = SentenceTransformerTrainingArguments(
    output_dir="vnese-biencoder-encoder-MNRL",
    
    num_train_epochs=14,
    per_device_train_batch_size=100,
    per_device_eval_batch_size=100,
    warmup_ratio=0.1,
    learning_rate=2e-5,
    fp16=True,
    bf16=False,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=100,
    logging_first_step=True,
    load_best_model_at_end=True,
    report_to="none",
    run_name="vnese-biencoder-encoder",
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset =val_dataset,
    loss=loss,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss
100,1.004500,0.630510
200,0.563200,0.512617
300,0.450800,0.515080
400,0.402800,0.483986
500,0.391100,0.504215
600,0.364300,0.508404
700,0.361600,0.492967
800,0.343200,0.489251
900,0.347900,0.499700


TrainOutput(global_step=900, training_loss=0.4704685612519582, metrics={'train_runtime': 2367.8755, 'train_samples_per_second': 47.53, 'train_steps_per_second': 0.479, 'total_flos': 0.0, 'train_loss': 0.4704685612519582, 'epoch': 11.11111111111111})

In [13]:
corpus_text = []

for raw in corpus:
    corpus_text.append(raw['content_Article'])

In [14]:
with open("/kaggle/input/privatetest/DRILL_PrivateTest/private_test.json","r",encoding = "utf-8") as f:
    file = json.load(f)
questions = []
idx_to_qid = {}
for idx,q in enumerate(file):
    output = rdrsegmenter.word_segment(q['question'])
    q['question'] = " ".join(output)
    questions.append(q['question'])
    idx_to_qid[idx] = q['qid']

In [15]:
corpus_embeddings = model.encode(corpus_text)

In [16]:
corpus_embeddings = corpus_embeddings.astype(np.float32)

In [17]:
!pip install -Uq faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.4 MB/s eta 0:00:00:00:0100:01


In [18]:
import faiss

dim = corpus_embeddings.shape[-1]

index = faiss.index_factory(dim, 'Flat', faiss.METRIC_INNER_PRODUCT)

In [19]:
index.train(corpus_embeddings)

In [20]:
print(index.is_trained)  

True


In [21]:
index.add(corpus_embeddings)

print(f"total number of vectors: {index.ntotal}")

total number of vectors: 97625


In [22]:
query_embeddings = model.encode(questions)

In [23]:
!pip install rank_bm25

In [24]:
from rank_bm25 import BM25Okapi

In [ ]:
top_k = 100
faiss_dists, faiss_ids = index.search(query_embeddings, k=top_k)

print(f"FAISS search completed: {len(faiss_ids)} queries, top {top_k} results each")

FAISS search completed: 627 queries, top 100 results each


In [ ]:
tokenized_corpus = [doc.split(" ") for doc in corpus_text]
bm25 = BM25Okapi(tokenized_corpus)

print(f"BM25 index created with {len(tokenized_corpus)} documents")

BM25 index created with 97625 documents


In [ ]:
bm25_results = []

for question in questions:
    tokenized_query = question.split(" ")
    doc_scores = bm25.get_scores(tokenized_query)
    
    top_100_indices = np.argsort(doc_scores)[::-1][:top_k]
    bm25_results.append(top_100_indices)

bm25_results = np.array(bm25_results)
print(f"BM25 search completed: {len(bm25_results)} queries, top {top_k} results each")

BM25 search completed: 627 queries, top 100 results each


In [ ]:
combined_results = []

for idx in range(len(questions)):
    faiss_indices = set(faiss_ids[idx].tolist())
    bm25_indices = set(bm25_results[idx].tolist())
    
    combined_indices = faiss_indices.union(bm25_indices)
    
    combined_indices = list(combined_indices)
    
    combined_results.append({
        'qid': idx_to_qid[idx],
        'question': questions[idx],
        'candidate_indices': combined_indices,
        'num_candidates': len(combined_indices)
    })

print(f"Combined results for {len(combined_results)} questions")
print(f"Average candidates per question: {np.mean([r['num_candidates'] for r in combined_results]):.1f}")

Combined results for 627 questions
Average candidates per question: 173.2


In [ ]:
cross_encoder_input = []

for result in combined_results:
    qid = result['qid']
    question = result['question']
    
    candidate_texts = [corpus_text[idx] for idx in result['candidate_indices']]
    
    cross_encoder_input.append({
        'qid': qid,
        'question': question,
        'candidates': candidate_texts,
        'num_candidates': len(candidate_texts)
    })

print(f"Prepared {len(cross_encoder_input)} questions for cross-encoder re-ranking")
print(f"Example - Question 0 has {cross_encoder_input[0]['num_candidates']} candidate documents")

Prepared 627 questions for cross-encoder re-ranking
Example - Question 0 has 184 candidate documents


In [ ]:
sample_idx = 0
print(f"Sample Question (qid: {cross_encoder_input[sample_idx]['qid']}):")
print(f"Question: {cross_encoder_input[sample_idx]['question'][:100]}...")
print(f"\nNumber of candidates: {cross_encoder_input[sample_idx]['num_candidates']}")
print(f"\nFirst candidate text:")
print(cross_encoder_input[sample_idx]['candidates'][1])

Sample Question (qid: 1497):
Question: Phạm_nhân không biết chữ có được tạo điều_kiện học văn_hoá nhằm xoá mù_chữ hay không ?...

Number of candidates: 184

First candidate text:
1 . Vận_động_viên đang học_tập tại các cơ_sở giáo_dục , cơ_sở giáo_dục nghề_nghiệp được triệu_tập vào đội_tuyển thể_thao quốc_gia , đội_tuyển thể_thao của các tỉnh , thành_phố trực_thuộc trung_ương , các ngành để tập_huấn và thi_đấu thì được cơ_quan sử_dụng vận_động_viên chi_trả học_phí theo quy_định của pháp_luật . 2 . Vận_động_viên đạt thành_tích xuất_sắc trong các giải thi_đấu thể_thao quốc_gia hoặc quốc_tế được xét đặc_cách tốt_nghiệp trung_học_cơ_sở , trung_học_phổ_thông nếu thời_gian thi trùng với thời_gian vận_động_viên tập_huấn ở nước_ngoài hoặc tham_dự thi_đấu tại các giải thể_thao quốc_tế . 3 . Cơ_quan sử_dụng vận_động_viên có trách_nhiệm như sau : a ) Tổ_chức học bổ_sung kiến_thức văn_hoá cho vận_động_viên thể_thao thành_tích cao sau khi vận_động_viên tham_dự tập_huấn , thi_đấu tại các giải thể_thao

In [ ]:
with open("cross_encoder_candidates.json", "w", encoding="utf-8") as f:
    json.dump(cross_encoder_input, f, ensure_ascii=False, indent=2)

print("Saved cross-encoder input to 'cross_encoder_candidates.json'")

Saved cross-encoder input to 'cross_encoder_candidates.json'


In [ ]:
ce_input_data = cross_encoder_input

In [35]:
answer = []
for q in ce_input_data:
    answer.append({
        'qid': q['qid'],
        'relevant_laws': list(set(int(corpus_Text2Id[t].split("_")[0]) for t in q['candidates']))
    })

with open(f"answer_top200.json", "w", encoding="utf-8") as f:
        json.dump(answer, f, ensure_ascii=False)

In [36]:
answer = []
for q in ce_input_data:
    answer.append({
        'qid': q['qid'],
        'relevant_laws': list(corpus_Text2Id[t] for t in q['candidates'])
    })

with open(f"answer_top200_input_rerank.json", "w", encoding="utf-8") as f:
        json.dump(answer, f, ensure_ascii=False)